# Where do agent runs get stuck?

Compare successful and failed runs as **sequences of tool events**, then count a complete retry pattern to check what the graph suggests.

This self-contained teaching example creates **16 synthetic runs**: six succeed directly, four recover after an error, four repeat the same error, and two fail at a permission check. It contains no real prompts, user data or credentials. The outcomes are assigned by the example generator, not inferred by Retentioneering. One path is one run.

Use Python 3.10–3.13. Run the cells in order. The package installation needs a network connection; the data itself is generated here. [Installation help](https://retentioneering.com/docs/installation) · [Current API](https://retentioneering.com/docs/eventstream)


In [ ]:
%pip install -q retentioneering

## 1. Make a small event log

Each row is one completed, sequential tool event. `outcome` labels the whole run for comparison. The `schema` declares `run_id` as the path ID, so one path is one run, and `outcome` as a segment column for comparisons; `event` and `timestamp` already use the default column names. We leave terminal labels such as `failed` and `succeeded` out of the event names so that the comparison can focus on the preceding behavior.


In [ ]:
import os

os.environ["RETENTIONEERING_NO_TRACK"] = "1"

import pandas as pd
import retentioneering as rete

scenarios = [
    (4, "failed", "retrieve plan tool:error retry tool:error retry tool:error"),
    (2, "failed", "retrieve permission:error"),
    (6, "succeeded", "retrieve plan tool:ok validate"),
    (4, "succeeded", "retrieve plan tool:error retry tool:ok validate"),
]
rows = []
run_number = 0
for count, outcome, path in scenarios:
    for _ in range(count):
        run_number += 1
        for step, event in enumerate(path.split()):
            rows.append(
                {
                    "run_id": f"run_{run_number:02d}",
                    "event": event,
                    "timestamp": pd.Timestamp("2026-01-01 10:00:00")
                    + pd.Timedelta(minutes=run_number - 1, seconds=step),
                    "outcome": outcome,
                }
            )
events = pd.DataFrame(rows)
stream = rete.Eventstream(
    events,
    schema={
        "path_cols": ["run_id"],
        "segment_cols": ["outcome"],
    },
)
events.head(8)

## 2. Compare paths

The groups have different sizes (six failed and ten successful runs), so use **average transitions per run**. The difference is failed minus succeeded: red means more transitions per failed run, blue means fewer. Click `tool:error` to focus on its connections.

A graph combines transitions across runs. Two visible edges alone do not establish that a complete route occurred in one run; we will check the pattern below.


In [ ]:
graph = stream.transition_graph(
    diff=("outcome", "failed", "succeeded"),
    edge_weight="avg_per_path",
)
graph

## 3. Align on the first error

The matrix aligns runs on `tool:error`. In this example, failed retrying runs encounter another error, while recovering runs move from retry to `tool:ok`. Runs with no `tool:error` are outside this anchored view—including the two permission failures. The next cell counts patterns over all runs to keep those denominators visible.


In [ ]:
stream.step_matrix(
    anchor="tool:error",
    max_steps=5,
    diff=("outcome", "failed", "succeeded"),
)

## 4. Check a complete retry pattern

Count runs containing `tool:error → retry → tool:error`. Each run counts once, even if the loop repeats. These counts should be **four of six failed runs and zero of ten successful runs** in the synthetic data.


In [ ]:
flags = (
    stream.get_metrics(
        [
            {
                "metric": "matches_pattern",
                "metric_args": {"pattern": "tool:error->retry->tool:error"},
            }
        ]
    )
    .iloc[:, 0]
    .rename("has_retry_loop")
)
outcomes = events.drop_duplicates("run_id").set_index("run_id")["outcome"]
summary = (
    flags.to_frame()
    .join(outcomes)
    .groupby("outcome")["has_retry_loop"]
    .agg(loop_runs="sum", all_runs="size")
)
summary

## 5. Inspect the numbers and share the result

`transition_graph_data()` uses the same parameters as the chart. Here its three tables are the difference, failed group and successful group. The exported HTML preserves viewing interactions; new calculations still require Python.


In [ ]:
difference, failed, succeeded = stream.transition_graph_data(
    diff=("outcome", "failed", "succeeded"),
    edge_weight="avg_per_path",
)
graph.export_html("agent-runs.html")
events.to_csv("agent-runs.csv", index=False)
difference.loc[["tool:error", "retry"], ["tool:error", "retry", "tool:ok"]]

## Bring your own traces

Replace the generated DataFrame with an export containing a run ID, a useful event label, a timestamp and an outcome. Any tracing platform works as long as you can export such a table; the example does not depend on a particular connector. Keep error and success states in the event names when that distinction matters. For nested or concurrent spans, first decide what constitutes an ordered step; simply sorting arbitrary parallel spans can create misleading transitions.

Then ask: Does the pattern recur across tools or versions? How many runs does it affect? Do retries recover? What other failure paths exist? The comparison describes associations in logged behavior. Test a proposed retry-policy change before concluding it improves outcomes.

[Path patterns](https://retentioneering.com/docs/path-patterns) · [Metrics](https://retentioneering.com/docs/path-metrics) · [Return to README](https://github.com/retentioneering/retentioneering-tools)
